In [3]:
import numpy as np
import pandas as pd
import openmatrix as omx
from pathlib import Path
from dbfread import DBF
import copy
from datetime import datetime
import os
import geopandas as gpd
from libpysal.weights import KNN


In [4]:
# import global TDM functions
import sys

sys.path.insert(0, "../Resources/2-Python/global-functions")
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()


## Inputs


In [5]:
path_inputs = Path.cwd() / "inputs"
path_outputs = Path.cwd() / "outputs"
path_start_files = path_inputs / "input_files/C28"

path_coeff_file = path_inputs / "coefficients_cal8.block"
path_se_file = path_start_files / "SE_File.dbf"
path_pa_file = path_start_files / "pa_initial.dbf"
path_tel_file = path_start_files / "telecommute.dbf"
path_hh_files = [
    path_start_files / f"HH{i}_PercTrips_segment_hbw.dbf" for i in range(1, 7)
]
path_taz_file = path_start_files / "WFv1000_TAZ.dbf"
path_taz_shpfile = path_start_files / "WFv1000_TAZ.shp"
path_dens_file = path_start_files / "IntersectionDensity.dbf"

# Skims and Logsums (assuming they are already converted to OMX in this folder)
path_skm_hwy = path_start_files / "skm_auto_Pk.omx"
path_skm_walk = path_start_files / "Best_Walk_Skims.omx"
path_logsums = path_start_files / "HBW_logsums_Pk.omx"
path_skm_transit_pk = path_start_files / "1Skm_TotTransitTime_Pk.omx"
path_skm_transit_ok = path_start_files / "1Skm_TotTransitTime_Ok.omx"

# We load this once here just to get the segment names for loading matrices
# initial_coeffs = load_block_coefficients(path_coeff_file)
# segments = list(initial_coeffs.keys())


In [25]:
used_zones = 3629
dummy_zones_str = "3563-3600"
external_zones_str = "3601-3629"

# read dbf files
df_se = pd.DataFrame(iter(DBF(path_se_file)))
df_pa = pd.DataFrame(iter(DBF(path_pa_file)))
df_tel = pd.DataFrame(iter(DBF(path_tel_file)))
df_hh = [pd.DataFrame(iter(DBF(f))) for f in path_hh_files]
df_taz = pd.DataFrame(iter(DBF(path_taz_file)))
gdf_taz = gpd.read_file(path_taz_shpfile)
df_dens = pd.DataFrame(iter(DBF(path_dens_file)))

# Create dummy/external mask
mask_external_dummys = np.zeros(used_zones, dtype=bool)
# apply_range_mask(mask_external_dummys, dummy_zones_str)
# apply_range_mask(mask_external_dummys, external_zones_str)


In [7]:
hts_hh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.hh"
).to_dataframe()
hts_person_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.person"
).to_dataframe()
hts_trip_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.trip_linked"
).to_dataframe()
hts_veh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.vehicle"
).to_dataframe()


c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


# Estimation Process 1


## Observed Data


In [8]:
hts_trip_23_merge = hts_trip_23.copy()
hts_trip_23_merge = hts_trip_23_merge[
    [
        "unique_id",
        "hh_id",
        "person_id",
        "vehicle_id",
        "pCO_TAZID_USTMv4",
        "aCO_TAZID_USTMv4",
        "pSUBAREAID",
        "aSUBAREAID",
        "PURP7_t",
        "trip_weight",
        "distance_miles",
    ]
]

# filter to HBW
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["PURP7_t"] == "HBW"]

# merge taz
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="pCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "pTAZID"}
)
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="aCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "aTAZID"}
)

# fitler to WF subarea
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["pSUBAREAID"] == 1]
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["aSUBAREAID"] == 1]

# vehicle ownership lookup table
vehown_lookup = (
    hts_veh_23.copy()
    .groupby("hh_id")["vehicle_id"]
    .count()
    .reset_index(name="veh_count")
)
vehown_lookup["vehown"] = vehown_lookup["veh_count"].clip(upper=2)
vehown_lookup = vehown_lookup[["hh_id", "vehown"]]

# income lookup table
income_lookup = hts_hh_23.copy()[["hh_id", "income_detailed"]]
income_lookup["income"] = np.select(
    [
        income_lookup["income_detailed"].isin([1, 2, 3, 4]),
        income_lookup["income_detailed"].isin([5, 6, 7, 8, 9, 10]),
    ],
    ["lo", "hi"],
    default="",
)

income_lookup = income_lookup[["hh_id", "income"]]

# merge vehown and income to trip table
hts_trip_23_merge = hts_trip_23_merge.merge(vehown_lookup, how="left", on="hh_id")
hts_trip_23_merge["vehown"] = hts_trip_23_merge["vehown"].fillna(0).astype(int)
hts_trip_23_merge = hts_trip_23_merge.merge(income_lookup, how="left", on="hh_id")

# calculate segment
hts_trip_23_merge["segment"] = np.where(
    hts_trip_23_merge["income"].notna() & (hts_trip_23_merge["income"] != ""),
    hts_trip_23_merge["vehown"].astype(str)
    + "veh_"
    + hts_trip_23_merge["income"].astype(str),
    "",
)

# filter to only known income trips
hts_trip_23_merge = hts_trip_23_merge[~(hts_trip_23_merge["segment"] == "")]

hts_trip_23_final = hts_trip_23_merge[
    ["vehown", "income", "pTAZID", "aTAZID", "trip_weight"]
]


## Model Data


### Retail, Industrial, and Other Employment PCT


In [ ]:
if "N" in df_se.columns:
    df_se_indexed = df_se.set_index(df_se["N"] - 1)
else:
    df_se_indexed = df_se.copy()

df_se_full = df_se_indexed.reindex(np.arange(used_zones)).fillna(0)
if "N" in df_se_full.columns:
    df_se_full["N"] = np.arange(1, used_zones + 1)

# ==========================================================
# DEFINE NEW AGGREGATE GROUPS
# ==========================================================
# Grouping 2: Trip Attraction
df_se_full["RETEMP2"] = df_se_full["RETL"] + df_se_full["FOOD"]
df_se_full["INDEMP2"] = df_se_full["MANU"] + df_se_full["WSLE"]
df_se_full["OFFEMP2"] = df_se_full["OFFI"] + df_se_full["GVED"] + df_se_full["HLTH"]
df_se_full["OTHEMP2"] = df_se_full["OTHR"]

# Grouping 3: Compensation
df_se_full["LOWEMP3"] = df_se_full["RETL"] + df_se_full["FOOD"]
df_se_full["MIDEMP3"] = (
    df_se_full["MANU"] + df_se_full["WSLE"] + df_se_full["GVED"] + df_se_full["OTHR"]
)
df_se_full["HIGHEMP3"] = df_se_full["OFFI"] + df_se_full["HLTH"]

# ==========================================================
# CALCULATE RATIOS / PERCENTAGES
# ==========================================================
# Original denom
denom_1 = (df_se_full["RETEMP"] + df_se_full["INDEMP"] + df_se_full["OTHEMP"]).values

# New denom (Summing all parts of Grouping 2 captures all 8 categories)
denom_new = (
    df_se_full["RETEMP2"]
    + df_se_full["INDEMP2"]
    + df_se_full["OFFEMP2"]
    + df_se_full["OTHEMP2"]
).values

# Original Ratios
retail_pct = np.divide(
    df_se_full["RETEMP"].values,
    denom_1,
    out=np.zeros_like(denom_1, dtype=float),
    where=denom_1 != 0,
)
ind_pct = np.divide(
    df_se_full["INDEMP"].values,
    denom_1,
    out=np.zeros_like(denom_1, dtype=float),
    where=denom_1 != 0,
)
other_pct = np.divide(
    df_se_full["OTHEMP"].values,
    denom_1,
    out=np.zeros_like(denom_1, dtype=float),
    where=denom_1 != 0,
)

# Grouping 2 Ratios (Trip Attraction)
retpct2 = np.divide(
    df_se_full["RETEMP2"].values,
    denom_new,
    out=np.zeros_like(denom_new, dtype=float),
    where=denom_new != 0,
)
indpct2 = np.divide(
    df_se_full["INDEMP2"].values,
    denom_new,
    out=np.zeros_like(denom_new, dtype=float),
    where=denom_new != 0,
)
offpct2 = np.divide(
    df_se_full["OFFEMP2"].values,
    denom_new,
    out=np.zeros_like(denom_new, dtype=float),
    where=denom_new != 0,
)
othpct2 = np.divide(
    df_se_full["OTHEMP2"].values,
    denom_new,
    out=np.zeros_like(denom_new, dtype=float),
    where=denom_new != 0,
)

# Grouping 3 Ratios (Compensation)
lowpct3 = np.divide(
    df_se_full["LOWEMP3"].values,
    denom_new,
    out=np.zeros_like(denom_new, dtype=float),
    where=denom_new != 0,
)
midpct3 = np.divide(
    df_se_full["MIDEMP3"].values,
    denom_new,
    out=np.zeros_like(denom_new, dtype=float),
    where=denom_new != 0,
)
highpct3 = np.divide(
    df_se_full["HIGHEMP3"].values,
    denom_new,
    out=np.zeros_like(denom_new, dtype=float),
    where=denom_new != 0,
)

# ==========================================================
# ASSEMBLE FINAL DATAFRAME
# ==========================================================
df_se_group = pd.DataFrame(
    {
        "TAZID": df_se_full["N"]
        if "N" in df_se_full.columns
        else np.arange(1, used_zones + 1),
        # Original Columns
        "RETEMP": df_se_full["RETEMP"],
        "INDEMP": df_se_full["INDEMP"],
        "OTHEMP": df_se_full["OTHEMP"],
        "RETPCT": retail_pct,
        "INDPCT": ind_pct,
        "OTHPCT": other_pct,
        # Grouping 2: Trip Attraction
        "RETEMP2": df_se_full["RETEMP2"],
        "INDEMP2": df_se_full["INDEMP2"],
        "OFFEMP2": df_se_full["OFFEMP2"],
        "OTHEMP2": df_se_full["OTHEMP2"],
        "RETPCT2": retpct2,
        "INDPCT2": indpct2,
        "OFFPCT2": offpct2,
        "OTHPCT2": othpct2,
        # Grouping 3: Compensation
        "LOWEMP3": df_se_full["LOWEMP3"],
        "MIDEMP3": df_se_full["MIDEMP3"],
        "HIGHEMP3": df_se_full["HIGHEMP3"],
        "LOWPCT3": lowpct3,
        "MIDPCT3": midpct3,
        "HIGHPCT3": highpct3,
    }
)

# Optional: Reset index if you want a clean 0-based integer index
df_se_group.reset_index(drop=True, inplace=True)

# Total employment (using the new denominator logic)
df_se_group["TOTEMP"] = denom_new
df_se_group


,TAZID,RETEMP,INDEMP,OTHEMP,RETPCT,INDPCT,OTHPCT,RETEMP2,INDEMP2,OFFEMP2,...,INDPCT2,OFFPCT2,OTHPCT2,LOWEMP3,MIDEMP3,HIGHEMP3,LOWPCT3,MIDPCT3,HIGHPCT3,TOTEMP
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,2.0,...,0.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3624,3625,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3625,3626,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3626,3627,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3627,3628,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Logsums by Segment


In [10]:
segments = ["0veh_lo", "0veh_hi", "1veh_lo", "1veh_hi", "2veh_lo", "2veh_hi"]
logsum_mtx = {}
with omx.open_file(path_logsums, "r") as f:
    for seg in segments:
        logsum_mtx[seg] = np.array(f[seg][:])


In [11]:
logsum_mtx["0veh_lo"]


array([[  2.64,   2.61,   0.11, ..., -21.95, -21.95, -21.95],
       [  2.61,   3.09,  -0.11, ..., -21.89, -21.89, -21.89],
       [  0.11,  -0.11,   2.59, ..., -21.13, -21.13, -21.13],
       ...,
       [ -0.77,  -0.77,  -0.77, ...,  -0.77,  -0.77,  -0.77],
       [ -0.77,  -0.77,  -0.77, ...,  -0.77,  -0.77,  -0.77],
       [ -0.77,  -0.77,  -0.77, ...,  -0.77,  -0.77,  -0.77]],
      shape=(3629, 3629))

### Highway Distance


In [12]:
with (
    omx.open_file(path_skm_hwy, "r") as skm_hwy,
    # omx.open_file(path_skm_walk, "r") as skm_walk,
):
    dist_mtx = np.array(skm_hwy["dist_GP"][:])
    # hwy_time = np.array(skm_hwy["ivt_GP"][:]) + np.array(skm_hwy["ovt"][:])
    # walk_gencost = np.array(skm_walk["GENCOST"][:])

dist_mtx


array([[  0.48,   0.5 ,  12.22, ..., 152.31, 132.78, 132.93],
       [  0.5 ,   0.23,  11.95, ..., 152.03, 132.5 , 132.65],
       [ 12.22,  11.95,   0.51, ..., 147.44, 127.91, 128.07],
       ...,
       [152.05, 151.77, 147.56, ...,   1.  ,  23.38,  23.52],
       [132.55, 132.27, 128.06, ...,  23.28,   1.  ,   2.2 ],
       [132.79, 132.51, 128.3 , ...,  23.52,   2.11,   1.  ]],
      shape=(3629, 3629))

### Skims


In [13]:
skims = logsum_mtx.copy()
skims["dist"] = dist_mtx

# 1. Determine dimensions (assuming N x N matrix)
n_rows, n_cols = skims["dist"].shape

# 2. Create the I and J columns (1-based indexing for TAZ IDs)
i_indices = np.repeat(np.arange(1, n_rows + 1), n_cols)
j_indices = np.tile(np.arange(1, n_cols + 1), n_rows)

# 3. Create a dictionary to hold the columns
data = {"I": i_indices, "J": j_indices}

# 4. Flatten each matrix in the skim dictionary and add it to the data
for key, matrix in skims.items():
    data[key] = matrix.flatten()

# 5. Create the final DataFrame
df_skim = pd.DataFrame(data)

# Optional: View the first few rows
df_skim


,I,J,0veh_lo,0veh_hi,1veh_lo,1veh_hi,2veh_lo,2veh_hi,dist
0,1,1,2.64,2.64,0.08,0.10,-0.09,-0.06,0.48
1,1,2,2.61,2.61,0.08,0.11,-0.08,-0.05,0.50
2,1,3,0.11,0.36,-2.17,-1.37,-2.31,-1.47,12.22
3,1,4,1.01,1.12,-2.04,-1.35,-2.28,-1.50,13.07
4,1,5,1.14,1.22,-0.38,-0.23,-0.47,-0.32,2.05
...,...,...,...,...,...,...,...,...,...
13169636,3629,3625,-0.77,-0.77,-0.77,-0.77,-0.77,-0.77,61.94
13169637,3629,3626,-0.77,-0.77,-0.77,-0.77,-0.77,-0.77,46.48
13169638,3629,3627,-0.77,-0.77,-0.77,-0.77,-0.77,-0.77,23.52
13169639,3629,3628,-0.77,-0.77,-0.77,-0.77,-0.77,-0.77,2.11


## Integrate Data


In [13]:
# -----------------------------
# LOAD DATA
# -----------------------------
trips = hts_trip_23_final.copy()
zones = df_se.copy()
zones = zones.rename(columns={"Z": "TAZID"})
skim = df_skim.copy()

# keep only zones with employment
zones = zones[zones["TOTEMP"] > 0].copy()

# -----------------------------
# CREATE ZONE LIST FOR SAMPLING
# -----------------------------
zones_ii = zones[zones["TAZID"] < 3563]
zone_ids = zones_ii["TAZID"].values


# -----------------------------
# SAMPLE FUNCTION
# -----------------------------
def sample_alternatives(row, n_sample=50):
    chosen = row["aTAZID"]

    # 1. Remove chosen from sampling pool
    pool = zone_ids[zone_ids != chosen]
    sampled = np.random.choice(pool, size=n_sample, replace=False)

    # 2. Include chosen alternative
    alts = np.append(sampled, chosen)

    # 3. Create a DataFrame from the ENTIRE row and repeat it 51 times
    # This preserves income, veh, trip_id, etc.
    df = pd.DataFrame([row] * len(alts))

    # 4. Overwrite the 'aTAZID' column with the sampled alternatives
    df["aTAZID"] = alts

    # 5. Mark choice
    df["CHOICE"] = (df["aTAZID"] == chosen).astype(int)

    return df


# -----------------------------
# BUILD CHOICE SET (The rest remains the same)
# -----------------------------
sampled_list = []
for _, row in trips.iterrows():
    sampled_list.append(sample_alternatives(row, 50))

choice_df = pd.concat(sampled_list, ignore_index=True)

# -----------------------------
# MERGE ZONAL DATA (destination side)
# -----------------------------
choice_df = choice_df.merge(zones, left_on="aTAZID", right_on="TAZID", how="left")

# -----------------------------
# MERGE SKIM
# -----------------------------
choice_df = choice_df.merge(
    skim, left_on=["pTAZID", "aTAZID"], right_on=["I", "J"], how="left"
)

# -----------------------------
# CREATE VARIABLES
# -----------------------------
# choice_df["dist"] = choice_df["dist"]
# choice_df["time"] = choice_df["TIME"]

choice_df["short_trip"] = 1 / np.maximum(1, choice_df["dist"])
choice_df["dist2"] = choice_df["dist"] ** 2
choice_df["dist3"] = choice_df["dist"] ** 3

choice_df["log_emp"] = np.log(choice_df["TOTEMP"])

# employment ratios
den = choice_df["RETEMP"] + choice_df["INDEMP"] + choice_df["OTHEMP"]

choice_df["retail_ratio"] = choice_df["RETEMP"] / den
choice_df["industrial_ratio"] = choice_df["INDEMP"] / den
choice_df["other_ratio"] = choice_df["OTHEMP"] / den

# intrazonal dummy
choice_df["intrazonal"] = (choice_df["pTAZID"] == choice_df["aTAZID"]).astype(int)

# unique observation id
choice_df["obs_id"] = np.repeat(np.arange(len(trips)), 51)

# -----------------------------
# SAVE FOR BIOGEME
# -----------------------------
choice_df


,vehown,income,pTAZID,aTAZID,trip_weight,CHOICE,TAZID,RETEMP,INDEMP,OTHEMP,...,dist,short_trip,dist2,dist3,log_emp,retail_ratio,industrial_ratio,other_ratio,intrazonal,obs_id
0,1,lo,2979.0,405.0,17.687357,0,405.0,713.0,251.0,681.0,...,78.24,0.012781,6121.4976,478945.972224,7.405496,0.433435,0.152584,0.413982,0,0
1,1,lo,2979.0,385.0,17.687357,0,385.0,59.0,195.0,150.0,...,80.31,0.012452,6449.6961,517975.093791,6.001415,0.146040,0.482673,0.371287,0,0
2,1,lo,2979.0,2866.0,17.687357,0,2866.0,342.0,34.0,1123.0,...,4.89,0.204499,23.9121,116.930169,7.312553,0.228152,0.022682,0.749166,0,0
3,1,lo,2979.0,403.0,17.687357,0,403.0,0.0,0.0,19.0,...,78.08,0.012807,6096.4864,476013.658112,2.944439,0.000000,0.000000,1.000000,0,0
4,1,lo,2979.0,2384.0,17.687357,0,2384.0,24.0,2.0,83.0,...,24.68,0.040519,609.1024,15032.647232,4.691348,0.220183,0.018349,0.761468,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707926,2,hi,316.0,16.0,0.000000,0,16.0,0.0,76.0,21.0,...,22.42,0.044603,502.6564,11269.556488,4.574711,0.000000,0.783505,0.216495,0,13880
707927,2,hi,316.0,2482.0,0.000000,0,2482.0,7.0,1.0,39.0,...,74.21,0.013475,5507.1241,408683.679461,3.850148,0.148936,0.021277,0.829787,0,13880
707928,2,hi,316.0,1441.0,0.000000,0,1441.0,14.0,154.0,133.0,...,45.00,0.022222,2025.0000,91125.000000,5.707110,0.046512,0.511628,0.441860,0,13880
707929,2,hi,316.0,1534.0,0.000000,0,1534.0,153.0,208.0,614.0,...,47.72,0.020956,2277.1984,108667.907648,6.882437,0.156923,0.213333,0.629744,0,13880


In [13]:
# 1. Construct the name of the column we want to pick from
# (e.g., 1 + "veh_" + "lo" -> "1veh_lo")
choice_df["target_col"] = (
    choice_df["vehown"].astype(int).astype(str) + "veh_" + choice_df["income"]
)

# 2. Initialize the logsum column with zeros or NaNs
choice_df["logsum"] = np.nan

# 3. Efficiently pick the values using a loop over the unique combinations
# (This is much faster than row-by-row iteration)
for col_name in choice_df["target_col"].unique():
    mask = choice_df["target_col"] == col_name
    if col_name in choice_df.columns:
        choice_df.loc[mask, "logsum"] = choice_df.loc[mask, col_name]

# 4. Clean up the helper column
choice_df.drop(columns=["target_col"], inplace=True)

choice_df.to_csv("choice_data.csv", index=False)


## Estimate Coefficients & Constants


- distance values look good. 0 veh low income is twice as sensitive as 2 veh high income which is what we would expect
- intrazonal values are mostly positive, because why wouldn't you want to stay in the same zone if you could
- logsums between 0.2 and .4 (should be between 0 and 1.0) -- changes to network will appropriately affect work location
- dist2 and dist3 super close to original values
- short trip factor is positive, seems fine
- _RED FLAG_ - magnitude of employment ratios are super high -- recommend change is to drop one of the values (other) and rerun


In [14]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("choice_data.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

varying_cols = [
    "dist",
    "dist2",
    "dist3",
    "short_trip",
    "log_emp",
    "logsum",
    "retail_ratio",
    "industrial_ratio",
    "other_ratio",
    "intrazonal",
]

# 🔥 include trip_weight here
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

segments = {
    "L0": (INCGRP == 0) * (VEHGRP == 0),
    "L1": (INCGRP == 0) * (VEHGRP == 1),
    "L2": (INCGRP == 0) * (VEHGRP == 2),
    "H0": (INCGRP == 1) * (VEHGRP == 0),
    "H1": (INCGRP == 1) * (VEHGRP == 1),
    "H2": (INCGRP == 1) * (VEHGRP == 2),
}

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
beta_short = Beta("beta_short", 0, None, None, 0)
beta_dist2 = Beta("beta_dist2", 0, None, None, 0)
beta_dist3 = Beta("beta_dist3", 0, None, None, 0)
beta_emp = Beta("beta_emp", 0, None, None, 0)


def create_segmented_beta(name, seg_dict):
    expr = 0
    for s_name, mask in seg_dict.items():
        b = Beta(f"{name}_{s_name}", 0, None, None, 0)
        expr += b * mask
    return expr


b_dist1 = create_segmented_beta("b_dist1", segments)
b_logsum = create_segmented_beta("b_logsum", segments)
b_iz = create_segmented_beta("b_iz", segments)
b_retail = create_segmented_beta("b_retail", segments)
b_ind = create_segmented_beta("b_ind", segments)

# ==================================================
# 5. UTILITY DICTIONARY (For 51 alternatives)
# ==================================================
V = {}
av = {}

for i in range(1, 52):
    V[i] = (
        beta_short * Variable(f"short_trip_{i}")
        + b_dist1 * Variable(f"dist_{i}")
        + beta_dist2 * Variable(f"dist2_{i}")
        + beta_dist3 * Variable(f"dist3_{i}")
        + beta_emp * Variable(f"log_emp_{i}")
        + b_logsum * Variable(f"logsum_{i}")
        + b_retail * Variable(f"retail_ratio_{i}")
        + b_ind * Variable(f"industrial_ratio_{i}")
        + b_iz * Variable(f"intrazonal_{i}")
    )
    av[i] = 1

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# 🔥 define weight variable
WEIGHT = Variable("trip_weight")

# 🔥 apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

biogeme.modelName = "destination_choice_wide_weighted"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


Starting Biogeme estimation...


c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Original data shape: (707931, 33)


C:\Users\cday\AppData\Local\Temp\ipykernel_21284\3470185549.py:136: DeprecationWarning: 'modelName' is deprecated. Please use 'model_name' instead.
  biogeme.modelName = "destination_choice_wide_weighted"


: 

: 

For some reason, I cannont exactly replicate the results that andy got from his estimation process. We think it has to do with different version of the biogeme package. Even so, the above code replicates it to the best we can. The model will use the results of Andy's code however, which are pasted below:


In [ ]:
# Name     Value  Robust std err.  Robust t-stat.  Robust p-value
# 0    b_dist1_L0 -0.214188         0.030389       -7.048123    1.813438e-12
# 1    b_dist1_L1 -0.069419         0.010021       -6.927662    4.278577e-12
# 2    b_dist1_L2 -0.070592         0.008842       -7.983930    1.332268e-15
# 3    b_dist1_H0 -0.129514         0.025426       -5.093671    3.511954e-07
# 4    b_dist1_H1 -0.067517         0.010916       -6.184896    6.214347e-10
# 5    b_dist1_H2 -0.052757         0.007833       -6.735538    1.633249e-11
# 6   b_logsum_L0  0.317888         0.087772        3.621759    2.926071e-04
# 7   b_logsum_L1  0.559290         0.069902        8.001039    1.332268e-15
# 8   b_logsum_L2  0.533347         0.063921        8.343878    0.000000e+00
# 9   b_logsum_H0  0.153432         0.118704        1.292558    1.961641e-01
# 10  b_logsum_H1  0.779985         0.113790        6.854612    7.150502e-12
# 11  b_logsum_H2  1.011520         0.082361       12.281590    0.000000e+00
# 12  b_retail_L0  0.391739         0.421638        0.929089    3.528428e-01
# 13  b_retail_L1 -1.781417         0.251721       -7.076945    1.473710e-12
# 14  b_retail_L2 -0.965687         0.278864       -3.462925    5.343371e-04
# 15  b_retail_H0 -0.550355         0.687232       -0.800828    4.232315e-01
# 16  b_retail_H1 -2.465867         0.219801      -11.218626    0.000000e+00
# 17  b_retail_H2 -1.681870         0.104727      -16.059631    0.000000e+00
# 18     b_ind_L0 -0.072970         0.417019       -0.174980    8.610952e-01
# 19     b_ind_L1 -1.361789         0.200760       -6.783180    1.175593e-11
# 20     b_ind_L2 -0.892342         0.232661       -3.835371    1.253749e-04
# 21     b_ind_H0  0.050213         0.511104        0.098244    9.217383e-01
# 22     b_ind_H1 -0.637994         0.119038       -5.359601    8.340573e-08
# 23     b_ind_H2 -0.332155         0.065409       -5.078106    3.812163e-07
# 24      b_iz_L0  0.227189         0.725415        0.313184    7.541406e-01
# 25      b_iz_L1  0.546081         0.346851        1.574399    1.153953e-01
# 26      b_iz_L2  1.365067         0.581207        2.348676    1.884027e-02
# 27      b_iz_H0  0.562778         1.041309        0.540452    5.888852e-01
# 28      b_iz_H1  0.293047         0.353470        0.829058    4.070715e-01
# 29      b_iz_H2  2.357390         0.211035       11.170634    0.000000e+00


# Estimation Process 2


## Data Prep


### Employment Ratio by Job Sector


In [32]:
denom = (
    df_se_full["RETL"]
    + df_se_full["FOOD"]
    + df_se_full["MANU"]
    + df_se_full["WSLE"]
    + df_se_full["OFFI"]
    + df_se_full["GVED"]
    + df_se_full["HLTH"]
    + df_se_full["OTHR"]
).values


# 2. Helper function to handle safe division across all sectors
def calc_pct(col_name):
    return np.divide(
        df_se_full[col_name].values,
        denom,
        out=np.zeros_like(denom, dtype=float),
        where=denom != 0,
    )


# 3. Generate the percentage arrays
retl_pct = calc_pct("RETL")
food_pct = calc_pct("FOOD")
manu_pct = calc_pct("MANU")
wsle_pct = calc_pct("WSLE")
offi_pct = calc_pct("OFFI")
gved_pct = calc_pct("GVED")
hlth_pct = calc_pct("HLTH")
othr_pct = calc_pct("OTHR")

# 4. Construct the new DataFrame
df_se_all1 = pd.DataFrame(
    {
        "TAZID": df_se_full["N"]
        if "N" in df_se_full.columns
        else np.arange(1, used_zones + 1),
        # Raw Counts
        "RETL": df_se_full["RETL"],
        "FOOD": df_se_full["FOOD"],
        "MANU": df_se_full["MANU"],
        "WSLE": df_se_full["WSLE"],
        "OFFI": df_se_full["OFFI"],
        "GVED": df_se_full["GVED"],
        "HLTH": df_se_full["HLTH"],
        "OTHR": df_se_full["OTHR"],
        # Percentages
        "retl_ratio": retl_pct,
        "food_ratio": food_pct,
        "manu_ratio": manu_pct,
        "wsle_ratio": wsle_pct,
        "offi_ratio": offi_pct,
        "gved_ratio": gved_pct,
        "hlth_ratio": hlth_pct,
        "othr_ratio": othr_pct,
    }
)
df_se_all1.reset_index(drop=True, inplace=True)

df_se_all = df_se_all1.copy()
df_se_all = df_se_all.merge(df_se_group, on="TAZID", how="left")
df_se_all


,TAZID,RETL,FOOD,MANU,WSLE,OFFI,GVED,HLTH,OTHR,retl_ratio,...,INDPCT2,OFFPCT2,OTHPCT2,LOWEMP3,MIDEMP3,HIGHEMP3,LOWPCT3,MIDPCT3,HIGHPCT3,TOTEMP
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3624,3625,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3625,3626,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3626,3627,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3627,3628,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
gdf_taz_copy = gdf_taz.copy()
df_se_copy = df_se_all1.copy()

# sectors
sector_cols = ["RETL", "FOOD", "MANU", "WSLE", "OFFI", "GVED", "HLTH", "OTHR"]

# totemp recalc
df_se_copy["TOTEMP"] = df_se_copy[sector_cols].sum(axis=1)

# merge columns
merge_cols = ["TAZID", "TOTEMP"] + sector_cols

# Merge into the spatial dataframe copy
gdf_taz_se = gdf_taz_copy.merge(df_se_copy[merge_cols], on="TAZID", how="left")

# Project to UTM Zone 12N (Utah) if necessary
if gdf_taz_se.crs is None:
    gdf_taz_se = gdf_taz_se.set_crs(epsg=4326)
if gdf_taz_se.crs.is_geographic:
    gdf_taz_se = gdf_taz_se.to_crs(epsg=26912)

# 2. Build the KNN weights (k=8 nearest neighbors)
weights = KNN.from_dataframe(gdf_taz_se, k=8)
neighbor_dict = weights.neighbors

# 3. Create a list of columns to calculate density for (Total + Sectors)
cols_to_calculate = ["TOTEMP"] + sector_cols

# 4. Calculate the smoothed values
job_densities = []

for idx, neighbors in neighbor_dict.items():
    # Combine the focal zone index with its 8 neighbors (Total = 9)
    group_indices = neighbors + [idx]
    group_gdf = gdf_taz_se.loc[group_indices]

    # Calculate Total Area in the cluster (sq miles)
    total_area = group_gdf.geometry.area.sum() / 2589988.0

    # Start the dictionary for this row
    taz_results = {"TAZID": gdf_taz_se.loc[idx, "TAZID"]}

    # Loop through TOTEMP and all individual sectors
    for col in cols_to_calculate:
        total_col_emp = group_gdf[col].sum()
        density = total_col_emp / total_area if total_area > 0 else 0
        taz_results[f"{col}_density"] = density

    job_densities.append(taz_results)

# 5. Merge the results back into our working spatial dataframe
results_df = pd.DataFrame(job_densities)
gdf_taz_se = gdf_taz_se.merge(results_df, on="TAZID")

# 6. View the final result cleanly
display_cols = ["TAZID", "TOTEMP_density"] + [f"{col}_density" for col in sector_cols]
taz_job_dens = gdf_taz_se[display_cols]
taz_job_dens


,TAZID,TOTEMP_density,RETL_density,FOOD_density,MANU_density,WSLE_density,OFFI_density,GVED_density,HLTH_density,OTHR_density
0,1.0,28.568452,0.000000,0.000000,8.484908,19.227269,0.000000,0.000000,0.155686,0.700589
1,2.0,24.952125,1.289320,0.000000,8.266813,11.679718,2.730324,0.000000,0.151685,0.834266
2,3.0,36.520922,0.148459,0.000000,8.091017,27.019544,0.000000,0.000000,0.148459,1.113443
3,4.0,30.921992,0.153841,0.000000,8.384321,21.076183,0.000000,0.000000,0.153841,1.153806
4,5.0,24.952125,1.289320,0.000000,8.266813,11.679718,2.730324,0.000000,0.151685,0.834266
...,...,...,...,...,...,...,...,...,...,...
3557,3127.0,60.568894,9.911274,0.275313,1.651879,1.101253,6.607516,7.158142,1.927192,31.936326
3558,3244.0,203.438163,35.495721,3.178721,3.178721,0.000000,18.542541,25.429770,17.482967,100.129721
3559,2369.0,60.261677,0.000000,0.379004,0.379004,0.000000,20.466230,37.142417,1.895021,0.000000
3560,2373.0,112.236392,2.416573,21.212141,0.268508,1.611049,24.702747,29.535893,13.425406,19.064076


### Intersection Density


In [34]:
df_int = df_dens.copy()
df_int = df_int[["TAZID", "IntDen"]]


### Parking Cost and CBD


In [46]:
df_taz_cbd_prk = df_taz.copy()
df_taz_cbd_prk = df_taz_cbd_prk[["TAZID", "CBD", "PRKCSTPERM", "PRKCSTTEMP"]]

# Replace 0 costs with a massive number so the utility tanks
# df_taz_cbd_prk["PRKCSTPERM"] = df_taz_cbd_prk["PRKCSTPERM"].replace(0, 9999)
# df_taz_cbd_prk["PRKCSTTEMP"] = df_taz_cbd_prk["PRKCSTTEMP"].replace(0, 9999)


### Transit Time


In [36]:
# The same helper function from before remains exactly the same
import numpy as np


def get_min_valid_time(matrix_list):
    stacked = np.stack(matrix_list)
    stacked_no_zeros = np.where(stacked == 0, np.inf, stacked)
    min_time = np.min(stacked_no_zeros, axis=0)
    return np.where(np.isinf(min_time), 0, min_time)


with (
    omx.open_file(path_skm_transit_pk, "r") as skm_transit_pk,
    omx.open_file(path_skm_transit_ok, "r") as skm_transit_ok,
):
    # 1. Local Transit (Mode 4) - Pulling from BOTH peak and off-peak
    local_keys = ["w4time", "d4time"]
    local_matrices = [np.array(skm_transit_pk[k][:]) for k in local_keys] + [
        np.array(skm_transit_ok[k][:]) for k in local_keys
    ]

    # 2. Premium Transit (Modes 5 through 9) - Pulling from BOTH peak and off-peak
    premium_keys = [
        "w5time",
        "w6time",
        "w7time",
        "w8time",
        "w9time",
        "d5time",
        "d6time",
        "d7time",
        "d8time",
        "d9time",
    ]
    premium_matrices = [np.array(skm_transit_pk[k][:]) for k in premium_keys] + [
        np.array(skm_transit_ok[k][:]) for k in premium_keys
    ]

    # 3. Calculate the absolute shortest valid times across the whole day
    local_min_time_daily = get_min_valid_time(local_matrices)
    premium_min_time_daily = get_min_valid_time(premium_matrices)


In [37]:
# 1. Determine dimensions (assuming the zone system is a square N x N matrix)
n_rows, n_cols = local_min_time_daily.shape

# 2. Create the I and J columns (1-based indexing for TAZ IDs)
i_indices = np.repeat(np.arange(1, n_rows + 1), n_cols)
j_indices = np.tile(np.arange(1, n_cols + 1), n_rows)

# 3. Flatten the matrices and create the final DataFrame directly
df_transit_skims = pd.DataFrame(
    {
        "I": i_indices,
        "J": j_indices,
        "lcl_transit_cost": local_min_time_daily.flatten(),
        "prm_transit_cost": premium_min_time_daily.flatten(),
    }
)

# Replace 0 costs with a massive number so the utility tanks
df_transit_skims["lcl_transit_cost"] = df_transit_skims["lcl_transit_cost"].replace(
    0, 5000
)
df_transit_skims["prm_transit_cost"] = df_transit_skims["prm_transit_cost"].replace(
    0, 5000
)


## Integrate Data


In [ ]:
print(df_se_group)


      TAZID  RETEMP  INDEMP  OTHEMP  RETPCT  INDPCT  OTHPCT  RETEMP2  INDEMP2  \
0         1     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   
1         2     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   
2         3     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   
3         4     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   
4         5     0.0     0.0     2.0     0.0     0.0     1.0      0.0      0.0   
...     ...     ...     ...     ...     ...     ...     ...      ...      ...   
3624   3625     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   
3625   3626     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   
3626   3627     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   
3627   3628     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   
3628   3629     0.0     0.0     0.0     0.0     0.0     0.0      0.0      0.0   

      OFFEMP2  ...  INDPCT2

In [48]:
# -----------------------------
# LOAD DATA
# -----------------------------
# trip data
trips = hts_trip_23_final.copy()

# zones level data
zones = df_se_group.copy()
zones = zones.rename(columns={"Z": "TAZID"})
zones = zones.merge(taz_job_dens, on="TAZID", how="left")
zones = zones.merge(df_int, on="TAZID", how="left")
zones = zones.merge(df_taz_cbd_prk, on="TAZID", how="left")

# skim data
dskim = df_skim.copy()
tskim = df_transit_skims.copy()
skim = dskim.merge(tskim, left_on=["I", "J"], right_on=["I", "J"], how="inner")

# keep only zones with employment
zones = zones[zones["TOTEMP"] > 0].copy()

# -----------------------------
# CREATE ZONE LIST FOR SAMPLING
# -----------------------------
zones_ii = zones[zones["TAZID"] < 3563]
zone_ids = zones_ii["TAZID"].values


# -----------------------------
# SAMPLE FUNCTION
# -----------------------------
def sample_alternatives(row, n_sample=19):
    chosen = row["aTAZID"]

    # 1. Remove chosen from sampling pool
    pool = zone_ids[zone_ids != chosen]
    sampled = np.random.choice(pool, size=n_sample, replace=False)

    # 2. Include chosen alternative
    alts = np.append(sampled, chosen)

    # 3. Create a DataFrame from the ENTIRE row and repeat it 20 times
    # This preserves income, veh, trip_id, etc.
    df = pd.DataFrame([row] * len(alts))

    # 4. Overwrite the 'aTAZID' column with the sampled alternatives
    df["aTAZID"] = alts

    # 5. Mark choice
    df["CHOICE"] = (df["aTAZID"] == chosen).astype(int)

    return df


# -----------------------------
# BUILD CHOICE SET (The rest remains the same)
# -----------------------------
sampled_list = []
for _, row in trips.iterrows():
    sampled_list.append(sample_alternatives(row, 19))

choice_df = pd.concat(sampled_list, ignore_index=True)

# -----------------------------
# MERGE ZONAL DATA (destination side)
# -----------------------------
choice_df = choice_df.merge(zones, left_on="aTAZID", right_on="TAZID", how="left")

# -----------------------------
# MERGE SKIM
# -----------------------------
choice_df = choice_df.merge(
    skim, left_on=["pTAZID", "aTAZID"], right_on=["I", "J"], how="left"
)

# -----------------------------
# CREATE VARIABLES
# -----------------------------

choice_df["short_trip"] = 1 / np.maximum(1, choice_df["dist"])
choice_df["dist2"] = choice_df["dist"] ** 2
choice_df["dist3"] = choice_df["dist"] ** 3

choice_df["log_emp"] = np.log(choice_df["TOTEMP"])

# intrazonal dummy
choice_df["intrazonal"] = (choice_df["pTAZID"] == choice_df["aTAZID"]).astype(int)

# unique observation id
choice_df["obs_id"] = np.repeat(np.arange(len(trips)), 20)

# -----------------------------
# SAVE FOR BIOGEME
# -----------------------------# 1. Construct the name of the column we want to pick from
# (e.g., 1 + "veh_" + "lo" -> "1veh_lo")
choice_df["target_col"] = (
    choice_df["vehown"].astype(int).astype(str) + "veh_" + choice_df["income"]
)

# 2. Initialize the logsum column with zeros or NaNs
choice_df["logsum"] = np.nan

# 3. Efficiently pick the values using a loop over the unique combinations
# (This is much faster than row-by-row iteration)
for col_name in choice_df["target_col"].unique():
    mask = choice_df["target_col"] == col_name
    if col_name in choice_df.columns:
        choice_df.loc[mask, "logsum"] = choice_df.loc[mask, col_name]

# 4. Clean up the helper column
choice_df.drop(columns=["target_col"], inplace=True)
choice_df.to_csv("outputs/choice_data2.csv", index=False)


## Estimate with New Values


- district-pair constants -> flag i-j pairs iwth value of 1 if they need something special
  - 21-16, 16-16, 14-14, 14-13,


**Estimation 1**


In [1]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("outputs/choice_data2.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# 🔥 FIX: Penalize 0 values to make unavailable transit impossible
df["lcl_transit_cost"] = df["lcl_transit_cost"].replace(0, 5000)
df["prm_transit_cost"] = df["prm_transit_cost"].replace(0, 5000)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

varying_cols = [
    "dist",
    "dist2",
    "dist3",
    "short_trip",
    "log_emp",
    "logsum",
    "intrazonal",
    "retl_ratio",
    "food_ratio",
    "manu_ratio",
    "wsle_ratio",
    "offi_ratio",
    "gved_ratio",
    "hlth_ratio",
    "othr_ratio",
    "RETL_density",
    "FOOD_density",
    "MANU_density",
    "WSLE_density",
    "OFFI_density",
    "GVED_density",
    "HLTH_density",
    "OTHR_density",
    "IntDen",
    "CBD",
    "PRKCSTPERM",
    "PRKCSTTEMP",
    "lcl_transit_cost",
    "prm_transit_cost",
]

# include trip_weight here
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Downcast to save RAM
float_cols = df_wide.select_dtypes(include=["float64"]).columns
df_wide[float_cols] = df_wide[float_cols].astype("float32")

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

segments = {
    "L0": (INCGRP == 0) * (VEHGRP == 0),
    "L1": (INCGRP == 0) * (VEHGRP == 1),
    "L2": (INCGRP == 0) * (VEHGRP == 2),
    "H0": (INCGRP == 1) * (VEHGRP == 0),
    "H1": (INCGRP == 1) * (VEHGRP == 1),
    "H2": (INCGRP == 1) * (VEHGRP == 2),
}

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
# Base Impedance & Spatial
beta_short = Beta("beta_short", 0, None, None, 0)
beta_dist2 = Beta("beta_dist2", 0, None, None, 0)
beta_dist3 = Beta("beta_dist3", 0, None, None, 0)
beta_emp = Beta("beta_emp", 0, None, None, 0)

# Densities & Urban Form
beta_intden = Beta("beta_intden", 0, None, None, 0)
beta_cbd = Beta("beta_cbd", 0, None, None, 0)

# Costs
beta_prk_perm = Beta("beta_prk_perm", 0, None, None, 0)
beta_prk_temp = Beta("beta_prk_temp", 0, None, None, 0)
beta_lcl_cost = Beta("beta_lcl_cost", 0, None, None, 0)
beta_prm_cost = Beta("beta_prm_cost", 0, None, None, 0)

# Unsegmented Lowercase SE Ratios (retl and manu are segmented below)
beta_food = Beta("beta_food", 0, None, None, 0)
beta_wsle = Beta("beta_wsle", 0, None, None, 0)
beta_offi = Beta("beta_offi", 0, None, None, 0)
beta_gved = Beta("beta_gved", 0, None, None, 0)
beta_hlth = Beta("beta_hlth", 0, None, None, 0)
beta_othr = Beta("beta_othr", 0, None, None, 0)

# Unsegmented Uppercase SE Densities (Changed to _den)
beta_RETL_den = Beta("beta_RETL_den", 0, None, None, 0)
beta_FOOD_den = Beta("beta_FOOD_den", 0, None, None, 0)
beta_MANU_den = Beta("beta_MANU_den", 0, None, None, 0)
beta_WSLE_den = Beta("beta_WSLE_den", 0, None, None, 0)
beta_OFFI_den = Beta("beta_OFFI_den", 0, None, None, 0)
beta_GVED_den = Beta("beta_GVED_den", 0, None, None, 0)
beta_HLTH_den = Beta("beta_HLTH_den", 0, None, None, 0)
beta_OTHR_den = Beta("beta_OTHR_den", 0, None, None, 0)


def create_segmented_beta(name, seg_dict):
    expr = 0
    for s_name, mask in seg_dict.items():
        b = Beta(f"{name}_{s_name}", 0, None, None, 0)
        expr += b * mask
    return expr


# Segmented parameters
b_dist1 = create_segmented_beta("b_dist1", segments)
b_logsum = create_segmented_beta("b_logsum", segments)
b_iz = create_segmented_beta("b_iz", segments)
b_retail = create_segmented_beta("b_retail", segments)  # Ties to retl_ratio
b_ind = create_segmented_beta("b_ind", segments)  # Ties to manu_ratio

# ==================================================
# 5. UTILITY DICTIONARY (For 20 alternatives)
# ==================================================
V = {}
av = {}
max_alts = df["alt_seq"].max()
for i in range(1, max_alts + 1):
    V[i] = (
        # Core Spatial & Impedance
        beta_short * Variable(f"short_trip_{i}")
        + b_dist1 * Variable(f"dist_{i}")
        + beta_dist2 * Variable(f"dist2_{i}")
        + beta_dist3 * Variable(f"dist3_{i}")
        + b_iz * Variable(f"intrazonal_{i}")
        + b_logsum * Variable(f"logsum_{i}")
        # Attraction Size
        + beta_emp * Variable(f"log_emp_{i}")
        # Lowercase SE Ratios
        + b_retail * Variable(f"retl_ratio_{i}")
        + beta_food * Variable(f"food_ratio_{i}")
        + b_ind * Variable(f"manu_ratio_{i}")
        + beta_wsle * Variable(f"wsle_ratio_{i}")
        + beta_offi * Variable(f"offi_ratio_{i}")
        + beta_gved * Variable(f"gved_ratio_{i}")
        + beta_hlth * Variable(f"hlth_ratio_{i}")
        + beta_othr * Variable(f"othr_ratio_{i}")
        # Uppercase SE Densities (Corrected to map to _density)
        + beta_RETL_den * Variable(f"RETL_density_{i}")
        + beta_FOOD_den * Variable(f"FOOD_density_{i}")
        + beta_MANU_den * Variable(f"MANU_density_{i}")
        + beta_WSLE_den * Variable(f"WSLE_density_{i}")
        + beta_OFFI_den * Variable(f"OFFI_density_{i}")
        + beta_GVED_den * Variable(f"GVED_density_{i}")
        + beta_HLTH_den * Variable(f"HLTH_density_{i}")
        + beta_OTHR_den * Variable(f"OTHR_density_{i}")
        # Density & Form
        + beta_intden * Variable(f"IntDen_{i}")
        + beta_cbd * Variable(f"CBD_{i}")
        # Costs
        + beta_prk_perm * Variable(f"PRKCSTPERM_{i}")
        + beta_prk_temp * Variable(f"PRKCSTTEMP_{i}")
        + beta_lcl_cost * Variable(f"lcl_transit_cost_{i}")
        + beta_prm_cost * Variable(f"prm_transit_cost_{i}")
    )
    av[i] = 1

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# define weight variable
WEIGHT = Variable("trip_weight")

# apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

biogeme.modelName = "destination_choice_wide_weighted"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())


Starting Biogeme estimation...


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Original data shape: (277620, 55)


C:\Users\cday\AppData\Local\Temp\ipykernel_39104\703065418.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_wide["CHOICE_VAR"] = chosen_alts
C:\Users\cday\AppData\Local\Temp\ipykernel_39104\703065418.py:219: DeprecationWarning: 'modelName' is deprecated. Please use 'model_name' instead.
  biogeme.modelName = "destination_choice_wide_weighted"
It seems that the optimization algorithm did not converge. Therefore, the results may not correspond to the maximum likelihood estimator. Check the specification of the model, or the criteria for convergence of the algorithm.


Estimated parameters:


C:\Users\cday\AppData\Local\Temp\ipykernel_39104\703065418.py:228: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  print(results.get_estimated_parameters())


             Name         Value  Robust std err.  Robust t-stat.  \
0      beta_short  1.838003e+00         0.207801        8.844993   
1      b_dist1_L0 -1.895646e-01         0.082537       -2.296721   
2      b_dist1_L1 -1.266562e-01         0.069551       -1.821067   
3      b_dist1_L2  1.178133e-02         0.028427        0.414448   
4      b_dist1_H0 -2.945726e-01         0.080369       -3.665269   
5      b_dist1_H1 -1.016973e-01         0.028411       -3.579516   
6      b_dist1_H2 -6.753915e-02         0.021007       -3.215145   
7      beta_dist2  1.192160e-04         0.000384        0.310644   
8      beta_dist3  1.886894e-06         0.000003        0.623071   
9         b_iz_L0  5.987292e-01         0.789975        0.757910   
10        b_iz_L1  1.752347e+00         1.040287        1.684484   
11        b_iz_L2  3.399230e+00         0.937961        3.624062   
12        b_iz_H0  2.878072e-01         1.915166        0.150278   
13        b_iz_H1  1.062563e+00         0.500932

**Estimation 2**


In [ ]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("outputs/choice_data2.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# 🔥 FIX: Penalize 0 values to make unavailable transit impossible
df["lcl_transit_cost"] = df["lcl_transit_cost"].replace(0, 5000)
df["prm_transit_cost"] = df["prm_transit_cost"].replace(0, 5000)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

# Removed excluded columns to save memory during the pivot
varying_cols = [
    "dist",
    "dist2",
    "dist3",
    "short_trip",
    "log_emp",
    "logsum",
    "intrazonal",
    "retl_ratio",
    "manu_ratio",
    "WSLE_density",
    "OFFI_density",
    "GVED_density",
    "HLTH_density",
    "PRKCSTTEMP",
    "lcl_transit_cost",
]

# include trip_weight here
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Downcast to save RAM
float_cols = df_wide.select_dtypes(include=["float64"]).columns
df_wide[float_cols] = df_wide[float_cols].astype("float32")

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

segments = {
    "L0": (INCGRP == 0) * (VEHGRP == 0),
    "L1": (INCGRP == 0) * (VEHGRP == 1),
    "L2": (INCGRP == 0) * (VEHGRP == 2),
    "H0": (INCGRP == 1) * (VEHGRP == 0),
    "H1": (INCGRP == 1) * (VEHGRP == 1),
    "H2": (INCGRP == 1) * (VEHGRP == 2),
}

# Create filtered segment dictionaries to exclude specific betas
segments_iz = {k: v for k, v in segments.items() if k not in ["L0", "H0"]}
segments_logsum = {
    k: v for k, v in segments.items() if k not in ["L0", "L1", "H0", "H1"]
}

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
# Base Impedance & Spatial
beta_short = Beta("beta_short", 0, None, None, 0)
beta_dist2 = Beta("beta_dist2", 0, None, None, 0)
beta_dist3 = Beta("beta_dist3", 0, None, None, 0)
beta_emp = Beta("beta_emp", 0, None, None, 0)

# Costs
beta_prk_temp = Beta("beta_prk_temp", 0, None, None, 0)
beta_lcl_cost = Beta("beta_lcl_cost", 0, None, None, 0)

# Unsegmented Uppercase SE Densities
beta_WSLE_den = Beta("beta_WSLE_den", 0, None, None, 0)
beta_OFFI_den = Beta("beta_OFFI_den", 0, None, None, 0)
beta_GVED_den = Beta("beta_GVED_den", 0, None, None, 0)
beta_HLTH_den = Beta("beta_HLTH_den", 0, None, None, 0)


def create_segmented_beta(name, seg_dict):
    expr = 0
    for s_name, mask in seg_dict.items():
        b = Beta(f"{name}_{s_name}", 0, None, None, 0)
        expr += b * mask
    return expr


# Segmented parameters
b_dist1 = create_segmented_beta("b_dist1", segments)
b_logsum = create_segmented_beta("b_logsum", segments_logsum)  # Uses filtered dict
b_iz = create_segmented_beta("b_iz", segments_iz)  # Uses filtered dict
b_retail = create_segmented_beta("b_retail", segments)
b_ind = create_segmented_beta("b_ind", segments)

# ==================================================
# 5. UTILITY DICTIONARY (For 20 alternatives)
# ==================================================
V = {}
av = {}
max_alts = df["alt_seq"].max()
for i in range(1, max_alts + 1):
    V[i] = (
        # Core Spatial & Impedance
        beta_short * Variable(f"short_trip_{i}")
        + b_dist1 * Variable(f"dist_{i}")
        + beta_dist2 * Variable(f"dist2_{i}")
        + beta_dist3 * Variable(f"dist3_{i}")
        + b_iz * Variable(f"intrazonal_{i}")
        + b_logsum * Variable(f"logsum_{i}")
        # Attraction Size
        + beta_emp * Variable(f"log_emp_{i}")
        # Lowercase SE Ratios
        + b_retail * Variable(f"retl_ratio_{i}")
        + b_ind * Variable(f"manu_ratio_{i}")
        # Uppercase SE Densities
        + beta_WSLE_den * Variable(f"WSLE_density_{i}")
        + beta_OFFI_den * Variable(f"OFFI_density_{i}")
        + beta_GVED_den * Variable(f"GVED_density_{i}")
        + beta_HLTH_den * Variable(f"HLTH_density_{i}")
        # Costs
        + beta_prk_temp * Variable(f"PRKCSTTEMP_{i}")
        + beta_lcl_cost * Variable(f"lcl_transit_cost_{i}")
    )
    av[i] = 1

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# define weight variable
WEIGHT = Variable("trip_weight")

# apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

# Renamed slightly so you don't overwrite your first run's HTML report
biogeme.modelName = "destination_choice_wide_weighted_v2"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())


Starting Biogeme estimation...
Original data shape: (277620, 55)


C:\Users\cday\AppData\Local\Temp\ipykernel_39104\1999879455.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_wide["CHOICE_VAR"] = chosen_alts
C:\Users\cday\AppData\Local\Temp\ipykernel_39104\1999879455.py:180: DeprecationWarning: 'modelName' is deprecated. Please use 'model_name' instead.
  biogeme.modelName = "destination_choice_wide_weighted_v2"
It seems that the optimization algorithm did not converge. Therefore, the results may not correspond to the maximum likelihood estimator. Check the specification of the model, or the criteria for convergence of the algorithm.


Estimated parameters:
             Name     Value  Robust std err.  Robust t-stat.  Robust p-value
0      beta_short  1.686622         0.183327        9.200093    0.000000e+00
1      b_dist1_L0 -0.225458         0.037727       -5.976101    2.285420e-09
2      b_dist1_L1 -0.097685         0.017275       -5.654809    1.560195e-08
3      b_dist1_L2  0.006863         0.027855        0.246374    8.053930e-01
4      b_dist1_H0 -0.223393         0.042975       -5.198212    2.012147e-07
5      b_dist1_H1 -0.106893         0.012826       -8.333945    0.000000e+00
6      b_dist1_H2 -0.064148         0.020186       -3.177790    1.484022e-03
7      beta_dist2  0.000123         0.000362        0.338287    7.351467e-01
8      beta_dist3  0.000002         0.000003        0.759106    4.477894e-01
9         b_iz_L1  1.224557         0.849313        1.441821    1.493529e-01
10        b_iz_L2  3.428101         1.025151        3.343998    8.258036e-04
11        b_iz_H1  0.904308         0.518572        1.

C:\Users\cday\AppData\Local\Temp\ipykernel_39104\1999879455.py:189: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  print(results.get_estimated_parameters())


In [ ]:
results.get_estimated_parameters().to_csv(
    "outputs/estimated_parameters2.csv", index=False
)


C:\Users\cday\AppData\Local\Temp\ipykernel_39104\3591037990.py:1: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  results.get_estimated_parameters().to_csv(


**Estimation 3**


In [ ]:
choice_df


,vehown,income,pTAZID,aTAZID,trip_weight,CHOICE,TAZID,RETEMP,INDEMP,OTHEMP,...,dist,lcl_transit_cost,prm_transit_cost,short_trip,dist2,dist3,log_emp,intrazonal,obs_id,logsum
0,1,lo,2979.0,1499.0,17.687357,0,1499.0,11.0,1.0,142.0,...,46.29,5000.00,5000.00,0.021603,2142.7641,99188.550189,5.036953,0,0,-8.21
1,1,lo,2979.0,2616.0,17.687357,0,2616.0,165.0,164.0,1113.0,...,17.26,66.08,62.89,0.057937,297.9076,5141.885176,7.273786,0,0,-3.05
2,1,lo,2979.0,826.0,17.687357,0,826.0,105.0,36.0,1672.0,...,54.98,5000.00,5000.00,0.018188,3022.8004,166193.565992,7.502738,0,0,-9.66
3,1,lo,2979.0,104.0,17.687357,0,104.0,0.0,1.0,118.0,...,97.20,5000.00,5000.00,0.010288,9447.8400,918330.048000,4.779123,0,0,-16.33
4,1,lo,2979.0,3052.0,17.687357,0,3052.0,113.0,100.0,443.0,...,5.44,34.92,40.14,0.183824,29.5936,160.989184,6.486161,0,0,-1.22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277615,2,hi,316.0,462.0,0.000000,0,462.0,0.0,0.0,36.0,...,10.66,5000.00,5000.00,0.093809,113.6356,1211.355496,3.583519,0,13880,-1.43
277616,2,hi,316.0,1582.0,0.000000,0,1582.0,29.0,31.0,167.0,...,49.04,5000.00,119.37,0.020392,2404.9216,117937.355264,5.424950,0,13880,-5.02
277617,2,hi,316.0,2909.0,0.000000,0,2909.0,130.0,24.0,644.0,...,83.29,5000.00,156.42,0.012006,6937.2241,577801.395289,6.682109,0,13880,-8.21
277618,2,hi,316.0,1179.0,0.000000,0,1179.0,12.0,24.0,22.0,...,44.60,5000.00,111.31,0.022422,1989.1600,88716.536000,4.060443,0,13880,-4.45


In [49]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("outputs/choice_data2.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# 🔥 FIX: Penalize 0 values to make unavailable transit impossible
# (Keeping this here in case your raw data still has these columns)
df["lcl_transit_cost"] = df["lcl_transit_cost"].replace(0, 5000)
df["prm_transit_cost"] = df["prm_transit_cost"].replace(0, 5000)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

# Removed all density columns to save memory during the pivot
varying_cols = [
    "dist",
    "short_trip",
    "log_emp",
    "logsum",
    "retl_ratio",
    "manu_ratio",
    "PRKCSTTEMP",
]

# include trip_weight here
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Downcast to save RAM
float_cols = df_wide.select_dtypes(include=["float64"]).columns
df_wide[float_cols] = df_wide[float_cols].astype("float32")

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

segments = {
    "L0": (INCGRP == 0) * (VEHGRP == 0),
    "L1": (INCGRP == 0) * (VEHGRP == 1),
    "L2": (INCGRP == 0) * (VEHGRP == 2),
    "H0": (INCGRP == 1) * (VEHGRP == 0),
    "H1": (INCGRP == 1) * (VEHGRP == 1),
    "H2": (INCGRP == 1) * (VEHGRP == 2),
}

# Create filtered segment dictionaries to exclude specific betas
segments_logsum = {
    k: v for k, v in segments.items() if k not in ["L0", "L1", "H0", "H1"]
}

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
# Base Impedance & Spatial
beta_short = Beta("beta_short", 0, None, None, 0)
beta_emp = Beta("beta_emp", 0, None, None, 0)

# Costs
beta_prk_temp = Beta("beta_prk_temp", 0, None, None, 0)


def create_segmented_beta(name, seg_dict):
    expr = 0
    for s_name, mask in seg_dict.items():
        b = Beta(f"{name}_{s_name}", 0, None, None, 0)
        expr += b * mask
    return expr


# Segmented parameters
b_dist1 = create_segmented_beta("b_dist1", segments)
b_logsum = create_segmented_beta("b_logsum", segments_logsum)  # Uses filtered dict
b_retail = create_segmented_beta("b_retail", segments)
b_ind = create_segmented_beta("b_ind", segments)

# ==================================================
# 5. UTILITY DICTIONARY (For 20 alternatives)
# ==================================================
V = {}
av = {}
max_alts = df["alt_seq"].max()
for i in range(1, max_alts + 1):
    V[i] = (
        # Core Spatial & Impedance
        beta_short * Variable(f"short_trip_{i}")
        + b_dist1 * Variable(f"dist_{i}")
        + b_logsum * Variable(f"logsum_{i}")
        # Attraction Size
        + beta_emp * Variable(f"log_emp_{i}")
        # Lowercase SE Ratios
        + b_retail * Variable(f"retl_ratio_{i}")
        + b_ind * Variable(f"manu_ratio_{i}")
        # Costs
        + beta_prk_temp * Variable(f"PRKCSTTEMP_{i}")
    )
    av[i] = 1

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# define weight variable
WEIGHT = Variable("trip_weight")

# apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

# Renamed to v4 to keep track of iterations
biogeme.modelName = "destination_choice_wide_weighted_v4"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())


Starting Biogeme estimation...
Original data shape: (277620, 59)


KeyError: "['retl_ratio', 'manu_ratio'] not in index"

In [2]:
results.get_estimated_parameters().to_csv(
    "outputs/estimated_parameters3.csv", index=False
)


C:\Users\cday\AppData\Local\Temp\ipykernel_30028\3968162011.py:1: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  results.get_estimated_parameters().to_csv(


**Estimation 4**


In [ ]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("outputs/choice_data2.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# 🔥 FIX: Penalize 0 values to make unavailable transit impossible
# (Keeping this here in case your raw data still has these columns)
df["lcl_transit_cost"] = df["lcl_transit_cost"].replace(0, 5000)
df["prm_transit_cost"] = df["prm_transit_cost"].replace(0, 5000)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

# Removed all density columns and old ratios.
# Added Grouping Level 2 variables.
varying_cols = [
    "dist",
    "short_trip",
    "log_emp",
    # "logsum",
    "RETPCT2",
    "INDPCT2",
    "OFFPCT2",
    # "PRKCSTTEMP",
    # "PRKCSTPERM",
]

# include trip_weight here
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Downcast to save RAM
float_cols = df_wide.select_dtypes(include=["float64"]).columns
df_wide[float_cols] = df_wide[float_cols].astype("float32")

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

segments = {
    "L0": (INCGRP == 0) * (VEHGRP == 0),
    "L1": (INCGRP == 0) * (VEHGRP == 1),
    "L2": (INCGRP == 0) * (VEHGRP == 2),
    "H0": (INCGRP == 1) * (VEHGRP == 0),
    "H1": (INCGRP == 1) * (VEHGRP == 1),
    "H2": (INCGRP == 1) * (VEHGRP == 2),
}

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
# Base Impedance & Spatial
beta_short = Beta("beta_short", 0, None, None, 0)
beta_emp = Beta("beta_emp", 0, None, None, 0)

# Costs
# beta_prk_temp = Beta("beta_prk_temp", 0, None, None, 0)
# beta_prk_perm = Beta("beta_prk_perm", 0, None, None, 0)


def create_segmented_beta(name, seg_dict):
    expr = 0
    for s_name, mask in seg_dict.items():
        b = Beta(f"{name}_{s_name}", 0, None, None, 0)
        expr += b * mask
    return expr


# Segmented parameters
b_dist1 = create_segmented_beta("b_dist1", segments)
# b_logsum = create_segmented_beta(
#    "b_logsum", segments
# )  # <--- Using the full segments dict now

# Level 2 Grouping Segmented Parameters
b_ret2 = create_segmented_beta("b_ret2", segments)
b_ind2 = create_segmented_beta("b_ind2", segments)
b_off2 = create_segmented_beta("b_off2", segments)

# ==================================================
# 5. UTILITY DICTIONARY (For 20 alternatives)
# ==================================================
V = {}
av = {}
# Casting max_alts to int to ensure range() works properly
max_alts = int(df["alt_seq"].max())
for i in range(1, max_alts + 1):
    V[i] = (
        # Core Spatial & Impedance
        beta_short * Variable(f"short_trip_{i}")
        + b_dist1 * Variable(f"dist_{i}")
        # + b_logsum * Variable(f"logsum_{i}")
        # Attraction Size
        + beta_emp * Variable(f"log_emp_{i}")
        # Level 2 SE Percentages (OTHPCT2 excluded as reference)
        + b_ret2 * Variable(f"RETPCT2_{i}")
        + b_ind2 * Variable(f"INDPCT2_{i}")
        + b_off2 * Variable(f"OFFPCT2_{i}")
        # Costs
        # + beta_prk_temp * Variable(f"PRKCSTTEMP_{i}")
        # + beta_prk_perm * Variable(f"PRKCSTPERM_{i}")
    )
    av[i] = 1

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# define weight variable
WEIGHT = Variable("trip_weight")

# apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

# Renamed to v4 to keep track of iterations
biogeme.modelName = "destination_choice_wide_weighted_v4"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())


Starting Biogeme estimation...
Original data shape: (277620, 59)


C:\Users\cday\AppData\Local\Temp\ipykernel_43512\4114368086.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_wide["CHOICE_VAR"] = chosen_alts
C:\Users\cday\AppData\Local\Temp\ipykernel_43512\4114368086.py:160: DeprecationWarning: 'modelName' is deprecated. Please use 'model_name' instead.
  biogeme.modelName = "destination_choice_wide_weighted_v4"


In [ ]:
results.get_estimated_parameters().to_csv(
    "outputs/estimated_parameters4.csv", index=False
)


C:\Users\cday\AppData\Local\Temp\ipykernel_23848\34565450.py:1: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  results.get_estimated_parameters().to_csv(


**Estimation 5**


In [1]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("outputs/choice_data2.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# 🔥 FIX: Penalize 0 values to make unavailable transit impossible
# (Keeping this here in case your raw data still has these columns)
df["lcl_transit_cost"] = df["lcl_transit_cost"].replace(0, 5000)
df["prm_transit_cost"] = df["prm_transit_cost"].replace(0, 5000)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

# Removed Level 2 groupings and added Level 3 Compensation variables
varying_cols = [
    "dist",
    "short_trip",
    "log_emp",  # <--- beta_emp still here!
    # "logsum",
    "LOWPCT3",
    "MIDPCT3",
    "HIGHPCT3",
    # "PRKCSTTEMP",
    # "PRKCSTPERM",
]

# include trip_weight here
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Downcast to save RAM
float_cols = df_wide.select_dtypes(include=["float64"]).columns
df_wide[float_cols] = df_wide[float_cols].astype("float32")

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

segments = {
    "L0": (INCGRP == 0) * (VEHGRP == 0),
    "L1": (INCGRP == 0) * (VEHGRP == 1),
    "L2": (INCGRP == 0) * (VEHGRP == 2),
    "H0": (INCGRP == 1) * (VEHGRP == 0),
    "H1": (INCGRP == 1) * (VEHGRP == 1),
    "H2": (INCGRP == 1) * (VEHGRP == 2),
}

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
# Base Impedance & Spatial
beta_short = Beta("beta_short", 0, None, None, 0)
beta_emp = Beta("beta_emp", 0, None, None, 0)

## Costs
# beta_prk_temp = Beta("beta_prk_temp", 0, None, None, 0)
# beta_prk_perm = Beta("beta_prk_perm", 0, None, None, 0)


def create_segmented_beta(name, seg_dict):
    expr = 0
    for s_name, mask in seg_dict.items():
        b = Beta(f"{name}_{s_name}", 0, None, None, 0)
        expr += b * mask
    return expr


# Segmented parameters
b_dist1 = create_segmented_beta("b_dist1", segments)
# b_logsum = create_segmented_beta(
#    "b_logsum", segments
# )  # <--- Using the full segments dict now

# Level 3 Compensation Segmented Parameters
b_mid3 = create_segmented_beta("b_mid3", segments)
b_high3 = create_segmented_beta("b_high3", segments)

# ==================================================
# 5. UTILITY DICTIONARY (For 20 alternatives)
# ==================================================
V = {}
av = {}
# Casting max_alts to int to ensure range() works properly
max_alts = int(df["alt_seq"].max())
for i in range(1, max_alts + 1):
    V[i] = (
        # Core Spatial & Impedance
        beta_short * Variable(f"short_trip_{i}")
        + b_dist1 * Variable(f"dist_{i}")
        # + b_logsum * Variable(f"logsum_{i}")
        # Attraction Size
        + beta_emp * Variable(f"log_emp_{i}")
        # Level 3 Compensation Percentages (LOWPCT3 excluded as reference)
        + b_mid3 * Variable(f"MIDPCT3_{i}")
        + b_high3 * Variable(f"HIGHPCT3_{i}")
        # Costs
        # + beta_prk_temp * Variable(f"PRKCSTTEMP_{i}")
        # + beta_prk_perm * Variable(f"PRKCSTPERM_{i}")
    )
    av[i] = 1

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# define weight variable
WEIGHT = Variable("trip_weight")

# apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

# Renamed to v5 to keep track of iterations
biogeme.modelName = "destination_choice_wide_weighted_v5"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())


Starting Biogeme estimation...


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Original data shape: (277620, 59)


C:\Users\cday\AppData\Local\Temp\ipykernel_43512\1888041305.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_wide["CHOICE_VAR"] = chosen_alts
C:\Users\cday\AppData\Local\Temp\ipykernel_43512\1888041305.py:157: DeprecationWarning: 'modelName' is deprecated. Please use 'model_name' instead.
  biogeme.modelName = "destination_choice_wide_weighted_v5"
It seems that the optimization algorithm did not converge. Therefore, the results may not correspond to the maximum likelihood estimator. Check the specification of the model, or the criteria for convergence of the algorithm.


Estimated parameters:
          Name     Value  Robust std err.  Robust t-stat.  Robust p-value
0   beta_short  2.014624         0.130950       15.384733    0.000000e+00
1   b_dist1_L0 -0.186252         0.036905       -5.046809    4.492509e-07
2   b_dist1_L1 -0.080608         0.011837       -6.809829    9.771517e-12
3   b_dist1_L2 -0.113654         0.008791      -12.927894    0.000000e+00
4   b_dist1_H0 -0.194880         0.030553       -6.378360    1.789948e-10
5   b_dist1_H1 -0.088429         0.004822      -18.338158    0.000000e+00
6   b_dist1_H2 -0.086644         0.002850      -30.402795    0.000000e+00
7     beta_emp  0.791896         0.024999       31.676607    0.000000e+00
8    b_mid3_L0 -0.093066         1.126460       -0.082618    9.341554e-01
9    b_mid3_L1  0.837747         0.560809        1.493820    1.352226e-01
10   b_mid3_L2 -0.131272         0.699119       -0.187767    8.510589e-01
11   b_mid3_H0 -0.215856         2.991971       -0.072145    9.424865e-01
12   b_mid3_H1  

C:\Users\cday\AppData\Local\Temp\ipykernel_43512\1888041305.py:166: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  print(results.get_estimated_parameters())


In [ ]:
results.get_estimated_parameters().to_csv(
    "outputs/estimated_parameters5.csv", index=False
)


C:\Users\cday\AppData\Local\Temp\ipykernel_30028\187598668.py:1: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  results.get_estimated_parameters().to_csv(
